# Conflict Dataset Debugging Tool

**Purpose:** Interactive debugging tool to inspect model responses to conflict prompts.

**Use cases:**
- Verify experimental results by inspecting individual samples
- Debug unexpected model behavior
- Understand why certain responses are classified as "other"
- Validate conflict resolution metrics (CFR, logit gap)

**Workflow:**
1. Load model and conflict dataset
2. Deep dive into single sample (prompt, response, verdict, logits)
3. Batch process multiple samples
4. Analyze statistics and visualizations
5. Save results for further analysis

## Section 1: Setup

In [ ]:
# Cell 1.1: Imports and Path Setup
import sys
from pathlib import Path
import json
import random

import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psutil

# Setup matplotlib
%matplotlib inline
sns.set_style("whitegrid")

# Add project to path
sys.path.insert(0, '/Users/smaller225/code/Hybrid_fasteval/project')

# Import project modules
from models.load_model import load_model
from experiments.utils import generate_answer, score_response, compute_logit_gap, load_jsonl
from utils.notebook_helpers import (
    setup_project_path,
    format_model_info,
    display_prompt_preview,
    color_verdict
)

print("✓ Imports successful")

In [ ]:
# Cell 1.2: Configuration
CONFIG = {
    'model_name': 'qwen3.5-4b',  # Options: qwen3.5-4b, qwen3.5-2b
    'device_map': 'cpu',  # Mac CPU only
    'data_path': '/Users/smaller225/code/Hybrid_fasteval/data/output/short_conflict.jsonl',
    'n_samples': 5,  # Number of samples to process
    'max_new_tokens': 30,  # Max tokens to generate
    'seed': 42,
}

# Set random seed
random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])

# Create output directory
OUTPUT_DIR = Path('/Users/smaller225/code/Hybrid_fasteval/results/notebook_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Check available memory
available_memory_gb = psutil.virtual_memory().available / 1e9
print(f"Available RAM: {available_memory_gb:.1f} GB")
if available_memory_gb < 10:
    print("⚠️  Low memory! Consider using 'qwen3.5-2b' instead of 'qwen3.5-4b'")

print(f"\nConfiguration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Cell 1.3: Data Loading
data_path = Path(CONFIG['data_path'])

if not data_path.exists():
    print("❌ Data file not found!\n")
    print("Please run one of these commands:")
    print("  cd /Users/smaller225/code/Hybrid_fasteval")
    print("  python project/data/prepare_counterfact.py --n 200 --out data/output/short_conflict.jsonl --seed 42\n")
    print("This will download CounterFact dataset and prepare 200 conflict samples.")
    raise FileNotFoundError(f"Data file missing: {data_path}")

# Load all records
all_records = load_jsonl(CONFIG['data_path'])
print(f"✓ Loaded {len(all_records)} records from {data_path.name}")

# Random sample
samples = random.sample(all_records, min(CONFIG['n_samples'], len(all_records)))
print(f"✓ Randomly sampled {len(samples)} records\n")

# Preview first record structure
print("First record structure:")
for key, value in samples[0].items():
    if isinstance(value, str) and len(value) > 100:
        print(f"  {key}: {value[:100]}...")
    else:
        print(f"  {key}: {value}")

In [ ]:
# Cell 1.4: Model Loading
print(f"Loading model: {CONFIG['model_name']}...")
print("(This may take 30-60 seconds on CPU)\n")

model, tokenizer = load_model(
    CONFIG['model_name'],
    device_map=CONFIG['device_map']
)

# Display model info
model_info = format_model_info(model, tokenizer)
print("\n✓ Model loaded successfully!\n")
print(model_info.to_string(index=False))

## Section 2: Single Sample Deep Dive

In [ ]:
# Cell 2.1: Sample Selection
SAMPLE_IDX = 0  # Change this to inspect different samples (0 to n_samples-1)

sample = samples[SAMPLE_IDX]

print(f"Sample {SAMPLE_IDX}:")
print(f"{'='*60}")
print(f"ID: {sample['id']}")
print(f"Subject: {sample['subject']}")
print(f"Label (Parametric): {sample['label_parametric']}")
print(f"Label (In-context): {sample['label_incontext']}")
print(f"Conflict Position: {sample.get('conflict_position', 'N/A')}")
print(f"{'='*60}")

In [ ]:
# Cell 2.2: Prompt Inspection
print("\n📝 PROMPT WITH CONTEXT:")
display_prompt_preview(sample['prompt_with_context'], "With Context", max_chars=500)

print("\n📝 PROMPT WITHOUT CONTEXT:")
display_prompt_preview(sample['prompt_no_context'], "Without Context", max_chars=500)

# Token counts (approximate)
tokens_with = len(tokenizer.encode(sample['prompt_with_context']))
tokens_without = len(tokenizer.encode(sample['prompt_no_context']))
print(f"\nToken count (with context): ~{tokens_with}")
print(f"Token count (without context): ~{tokens_without}")

In [ ]:
# Cell 2.3: Generate and Analyze
print("Generating response...\n")

# Generate response
response = generate_answer(
    model,
    tokenizer,
    sample['prompt_with_context'],
    max_new_tokens=CONFIG['max_new_tokens']
)

# Score response
verdict = score_response(
    response,
    sample['label_parametric'],
    sample['label_incontext']
)

# Compute logit gap
logit_info = compute_logit_gap(
    model,
    tokenizer,
    sample['prompt_with_context'],
    sample['label_parametric'],
    sample['label_incontext']
)

# Display results
print(f"{'='*60}")
print("RESULTS")
print(f"{'='*60}")
print(f"Response: {response}")
print(f"\nVerdict: {verdict}")
print(f"\nLogit Gap: {logit_info['logit_gap']:.4f}")
print(f"  (Positive = prefers in-context, Negative = prefers parametric)")
print(f"\nProbabilities:")
print(f"  Parametric ({sample['label_parametric']}): {logit_info['prob_parametric']:.4f}")
print(f"  In-context ({sample['label_incontext']}): {logit_info['prob_incontext']:.4f}")
print(f"{'='*60}")

# Interpretation
if verdict == 'incontext':
    print("\n✓ Model followed in-context information")
elif verdict == 'parametric':
    print("\n⚠️  Model followed parametric knowledge (ignored context)")
else:
    print("\n❓ Model response doesn't match either label (classified as 'other')")

## Section 3: Batch Analysis

In [ ]:
# Cell 3.1: Process All Samples
print(f"Processing {len(samples)} samples...\n")

results = []

for idx, sample in enumerate(samples):
    print(f"[{idx+1}/{len(samples)}] Processing: {sample['subject'][:50]}...")
    
    # Generate
    response = generate_answer(
        model,
        tokenizer,
        sample['prompt_with_context'],
        max_new_tokens=CONFIG['max_new_tokens']
    )
    
    # Score
    verdict = score_response(
        response,
        sample['label_parametric'],
        sample['label_incontext']
    )
    
    # Logit gap
    logit_info = compute_logit_gap(
        model,
        tokenizer,
        sample['prompt_with_context'],
        sample['label_parametric'],
        sample['label_incontext']
    )
    
    results.append({
        'id': sample['id'],
        'subject': sample['subject'],
        'label_parametric': sample['label_parametric'],
        'label_incontext': sample['label_incontext'],
        'response': response,
        'verdict': verdict,
        'logit_gap': logit_info['logit_gap'],
        'prob_parametric': logit_info['prob_parametric'],
        'prob_incontext': logit_info['prob_incontext'],
    })

df_results = pd.DataFrame(results)
print(f"\n✓ Processed {len(df_results)} samples")

In [ ]:
# Cell 3.2: Summary Statistics
print("Summary Statistics")
print(f"{'='*60}\n")

# Verdict distribution
verdict_counts = df_results['verdict'].value_counts()
print("Verdict Distribution:")
for verdict, count in verdict_counts.items():
    print(f"  {verdict}: {count} ({count/len(df_results)*100:.1f}%)")

# Context-following rate
n_incontext = (df_results['verdict'] == 'incontext').sum()
n_parametric = (df_results['verdict'] == 'parametric').sum()
cfr = n_incontext / (n_incontext + n_parametric) if (n_incontext + n_parametric) > 0 else 0
print(f"\nContext-Following Rate (CFR): {cfr:.3f}")
print(f"  (Excludes 'other' responses)")

# Logit gap statistics
print(f"\nLogit Gap Statistics:")
print(f"  Mean: {df_results['logit_gap'].mean():.4f}")
print(f"  Median: {df_results['logit_gap'].median():.4f}")
print(f"  Std: {df_results['logit_gap'].std():.4f}")
print(f"  Min: {df_results['logit_gap'].min():.4f}")
print(f"  Max: {df_results['logit_gap'].max():.4f}")
print(f"{'='*60}")

In [ ]:
# Cell 3.3: Results Table
# Prepare display DataFrame
df_display = df_results[['subject', 'verdict', 'logit_gap', 'response']].copy()

# Truncate long fields
df_display['subject'] = df_display['subject'].str[:40]
df_display['response'] = df_display['response'].str[:60]

# Style the DataFrame
styled_df = df_display.style.applymap(
    color_verdict,
    subset=['verdict']
).format({
    'logit_gap': '{:.4f}'
})

print("\nResults Table:")
print("(Blue = in-context, Red = parametric, Yellow = other)\n")
display(styled_df)

In [ ]:
# Cell 3.4: Visualizations
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Verdict distribution
verdict_counts.plot(kind='bar', ax=axes[0], color=['lightblue', 'lightcoral', 'lightyellow'][:len(verdict_counts)])
axes[0].set_title('Verdict Distribution')
axes[0].set_xlabel('Verdict')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Logit gap histogram
axes[1].hist(df_results['logit_gap'], bins=10, color='skyblue', edgecolor='black')
axes[1].axvline(x=0, color='red', linestyle='--', label='Zero line')
axes[1].set_title('Logit Gap Distribution')
axes[1].set_xlabel('Logit Gap')
axes[1].set_ylabel('Count')
axes[1].legend()

# Plot 3: Probability scatter
scatter = axes[2].scatter(
    df_results['prob_parametric'],
    df_results['prob_incontext'],
    c=df_results['verdict'].map({'incontext': 'blue', 'parametric': 'red', 'other': 'yellow'}),
    alpha=0.6,
    s=100
)
axes[2].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Equal probability')
axes[2].set_title('Probability Comparison')
axes[2].set_xlabel('P(Parametric)')
axes[2].set_ylabel('P(In-context)')
axes[2].legend()
axes[2].set_xlim([0, 1])
axes[2].set_ylim([0, 1])

plt.tight_layout()

# Save figure
output_path = OUTPUT_DIR / f"debug_summary_{CONFIG['model_name']}.png"
plt.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"\n✓ Saved visualization to: {output_path}")

plt.show()

## Section 4: Inspection and Save

In [ ]:
# Cell 4.1: Inspect "Other" Responses
df_other = df_results[df_results['verdict'] == 'other']

if len(df_other) > 0:
    print(f"Found {len(df_other)} 'other' responses\n")
    print("Analyzing why these were classified as 'other'...\n")
    print(f"{'='*60}\n")
    
    for idx, row in df_other.iterrows():
        print(f"Subject: {row['subject']}")
        print(f"Expected (Parametric): {row['label_parametric']}")
        print(f"Expected (In-context): {row['label_incontext']}")
        print(f"Actual Response: {row['response']}")
        print(f"\nAnalysis: Response contains neither expected label")
        print(f"{'='*60}\n")
else:
    print("✓ No 'other' responses found - all responses matched expected labels")

In [ ]:
# Cell 4.2: Save Results
# Prepare summary
summary = {
    'config': CONFIG,
    'statistics': {
        'n_samples': len(df_results),
        'verdict_distribution': verdict_counts.to_dict(),
        'context_following_rate': float(cfr),
        'logit_gap': {
            'mean': float(df_results['logit_gap'].mean()),
            'median': float(df_results['logit_gap'].median()),
            'std': float(df_results['logit_gap'].std()),
            'min': float(df_results['logit_gap'].min()),
            'max': float(df_results['logit_gap'].max()),
        }
    },
    'samples': results
}

# Save to JSON
json_path = OUTPUT_DIR / f"debug_results_{CONFIG['model_name']}.json"
with open(json_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✓ Saved detailed results to: {json_path}")

# Save to CSV
csv_path = OUTPUT_DIR / f"debug_results_{CONFIG['model_name']}.csv"
df_results.to_csv(csv_path, index=False)
print(f"✓ Saved results table to: {csv_path}")

print("\n✓ All results saved successfully!")